In [3]:
import gymnasium as gym
import numpy as np

from huggingface_sb3 import load_from_hub
from stable_baselines3 import DQN


checkpoint = load_from_hub(
    repo_id="sb3/dqn-MountainCar-v0",
    filename="dqn-MountainCar-v0.zip",
)

env = gym.make("MountainCar-v0")

model = DQN.load(
    checkpoint,
    env=env,
    custom_objects={
        "observation_space": env.observation_space,
        "action_space": env.action_space,
    },
)

returns = []
episode_lengths = []
successes = []

for seed in range(100):
    obs, info = env.reset(seed=seed)
    terminated = False
    truncated = False
    episode_return = 0.0
    episode_length = 0

    while not (terminated or truncated):
        action, _ = model.predict(obs, deterministic=True)

        # Convert SB3's NumPy output to a scalar integer.
        action = int(np.asarray(action).item())

        obs, reward, terminated, truncated, info = env.step(action)

        episode_return += float(reward)
        episode_length += 1

    returns.append(episode_return)
    episode_lengths.append(episode_length)
    successes.append(terminated)

env.close()

print(f"Mean return: {np.mean(returns):.2f} ± {np.std(returns):.2f}")
print(f"Mean episode length: {np.mean(episode_lengths):.2f}")
print(f"Success rate: {np.mean(successes):.1%}")

/vol/bitbucket/ma5923/_projects/CertifiedContinualLearning/.venv/lib/python3.10/site-packages/stable_baselines3/common/save_util.py:167: UserWarning: Could not deserialize object learning_rate. Consider using `custom_objects` argument to replace this object.
Exception: 'bytes' object cannot be interpreted as an integer
  warnings.warn(
/vol/bitbucket/ma5923/_projects/CertifiedContinualLearning/.venv/lib/python3.10/site-packages/stable_baselines3/common/save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: 'bytes' object cannot be interpreted as an integer
  warnings.warn(
/vol/bitbucket/ma5923/_projects/CertifiedContinualLearning/.venv/lib/python3.10/site-packages/stable_baselines3/common/save_util.py:167: UserWarning: Could not deserialize object exploration_schedule. Consider using `custom_objects` argument to replace this object.
Exception: 'bytes' object cannot be interpreted as an intege

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Mean return: -100.02 ± 9.50
Mean episode length: 100.02
Success rate: 100.0%


### PPO

In [5]:
import gymnasium as gym
import numpy as np

from huggingface_sb3 import load_from_hub
from stable_baselines3 import PPO


# Create the Gymnasium environment first so its spaces can replace
# the legacy OpenAI Gym spaces stored in the checkpoint.
env = gym.make("MountainCar-v0")

checkpoint = load_from_hub(
    repo_id="vukpetar/ppo-MountainCar-v1",
    filename="ppo-mountaincar-v0.zip",
)

model = PPO.load(
    checkpoint,
    device="cpu",
    custom_objects={
        "observation_space": env.observation_space,
        "action_space": env.action_space,
    },
)

returns = []
episode_lengths = []
successes = []

for seed in range(100):
    obs, info = env.reset(seed=seed)

    terminated = False
    truncated = False
    episode_return = 0.0
    episode_length = 0

    while not (terminated or truncated):
        # Deterministic evaluation chooses the highest-probability action.
        action, _ = model.predict(obs, deterministic=True)
        action = int(np.asarray(action).item())

        obs, reward, terminated, truncated, info = env.step(action)

        episode_return += float(reward)
        episode_length += 1

    returns.append(episode_return)
    episode_lengths.append(episode_length)
    successes.append(terminated)

env.close()

returns = np.asarray(returns)
episode_lengths = np.asarray(episode_lengths)
successes = np.asarray(successes)

print(f"Mean return: {returns.mean():.2f} ± {returns.std():.2f}")
print(f"Mean episode length: {episode_lengths.mean():.2f}")
print(f"Success rate: {successes.mean():.1%}")
print(f"Best return: {returns.max():.0f}")
print(f"Worst return: {returns.min():.0f}")

ppo-mountaincar-v0.zip:   0%|          | 0.00/133k [00:00<?, ?B/s]

Mean return: -97.36 ± 7.54
Mean episode length: 97.36
Success rate: 100.0%
Best return: -83
Worst return: -105
